## Preprocesamiento de Datos MIMIC-III

Carga, limpieza y transformación de datos clínicos antes del modelado.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
# Cargar tablas MIMIC-III
base_path = 'C:/Users/Ines/Desktop/Nova_entrega/Parte_topicos/'

icustays_cleaned = pd.read_excel(f'{base_path}icustays_cleaned.xlsx')
admissions_cleaned = pd.read_excel(f'{base_path}admissions_cleaned.xlsx')
diagnoses_icd_cleaned = pd.read_excel(f'{base_path}diagnoses_icd_cleaned.xlsx')
d_items = pd.read_csv(f'{base_path}D_ITEMS.csv')
patients_cleaned = pd.read_excel(f'{base_path}patients_cleaned.xlsx')
chartevents = pd.read_csv(f'{base_path}CHARTEVENTS.csv')

In [ ]:
# Limpiar d_items: eliminar columnas innecesarias y valores nulos
d_items = d_items.drop(columns=['ROW_ID','ABBREVIATION','DBSOURCE','LINKSTO', 'CATEGORY','UNITNAME','PARAM_TYPE', 'CONCEPTID'])
d_items_cleaned = d_items.dropna(subset=['LABEL'])

In [ ]:
# Filtrar pacientes con enfermedades del sistema circulatorio (recode == 6)
subject_ids_recode_6 = diagnoses_icd_cleaned.loc[diagnoses_icd_cleaned['recode'] == 6, 'SUBJECT_ID'].unique()

patients_filtred = patients_cleaned[patients_cleaned['SUBJECT_ID'].isin(subject_ids_recode_6)]
patients_filtred = patients_filtred.drop(columns=['ROW_ID','DOB','DOD'])

In [ ]:
# Limpiar chartevents: eliminar columnas innecesarias
chartevents = chartevents.drop(columns=['ROW_ID','STORETIME', 'CGID', 'VALUENUM','VALUEUOM','WARNING', 'ERROR', 'RESULTSTATUS', 'STOPPED'])
subject_ids_filtrados = patients_filtred['SUBJECT_ID'].unique()

chartevents_filtred = chartevents[chartevents['SUBJECT_ID'].isin(subject_ids_filtrados)]

In [ ]:
# Fusionar datos: chartevents con etiquetas, información de UCI y admisiones
chartevents_filt_com_label = chartevents_filtred.merge(
    d_items_cleaned[['ITEMID', 'LABEL']], 
    on='ITEMID',
    how='left'
)

chartevents_filt_com_icu = chartevents_filt_com_label.merge(
    icustays_cleaned[['INTIME','LOS','ICUSTAY_ID']],
    on='ICUSTAY_ID',
    how='left'
)

df_final = chartevents_filt_com_icu.merge(
    admissions_cleaned[['HADM_ID','DIAGNOSIS','HAS_CHARTEVENTS_DATA','DEATHTIME','ADMISSION_TYPE','ETHNICITY','HOSPITAL_EXPIRE_FLAG']],
    on='HADM_ID',
    how='left'
)

In [ ]:
# Eliminar filas con valores faltantes en HAS_CHARTEVENTS_DATA
df_final = df_final[df_final['HAS_CHARTEVENTS_DATA'].notna()]
df_final = df_final.drop(columns=['HAS_CHARTEVENTS_DATA'])

In [ ]:
# Convertir VALUE a numérico (coercionar no convertibles a NaN)
df_final['VALUE'] = pd.to_numeric(df_final['VALUE'], errors='coerce')

# Rellenar valores faltantes con la media por ITEMID
mean_values = df_final.groupby('ITEMID')['VALUE'].mean()
df_final['VALUE'] = df_final['VALUE'].fillna(df_final['ITEMID'].map(mean_values))

In [ ]:
# Convertir fechas al formato datetime
df_final['CHARTTIME'] = pd.to_datetime(df_final['CHARTTIME'], errors='coerce')
df_final['INTIME'] = pd.to_datetime(df_final['INTIME'], errors='coerce')

# Eliminar filas con fechas faltantes
df_final = df_final.dropna(subset=['CHARTTIME', 'INTIME'])

In [ ]:
# Codificar variables categóricas
categorical_cols = ['LABEL', 'DIAGNOSIS', 'ETHNICITY', 'ADMISSION_TYPE']

for col in categorical_cols:
    le = LabelEncoder()
    df_final[col + '_ENCODED'] = le.fit_transform(df_final[col])

# Eliminar columnas categóricas originales
df_final = df_final.drop(columns=categorical_cols)

In [ ]:
# Crear características agregadas por ICUSTAY_ID
df_grouped = df_final.groupby(['ICUSTAY_ID', 'ITEMID'])['VALUE'].mean()
df_pivot = df_grouped.unstack()

# Eliminar columnas con valores constantes
nunique_per_column = df_pivot.nunique(dropna=True)
cols_to_keep = nunique_per_column[nunique_per_column > 1].index
df_pivot_filtered = df_pivot[cols_to_keep]

# Rellenar valores faltantes con la media de cada columna
df_pivot_filtered = df_pivot_filtered.fillna(df_pivot_filtered.mean())

In [ ]:
# Fusionar con Length of Stay (LOS)
df_modelo = df_pivot_filtered.merge(
    icustays_cleaned[['ICUSTAY_ID', 'LOS']], 
    on='ICUSTAY_ID', 
    how='inner'
)

In [ ]:
# Filtrar para primeras 24 horas después de admisión
df_final['hours_since_admit'] = (df_final['CHARTTIME'] - df_final['INTIME']).dt.total_seconds() / 3600
df_24h = df_final[df_final['hours_since_admit'] <= 24].copy()

# Agrupar datos para modelo
df_24h_grouped = df_24h.groupby(['ICUSTAY_ID', 'ITEMID'])['VALUE'].mean()
df_24h_pivot = df_24h_grouped.unstack()

# Eliminar columnas constantes
nunique_24h = df_24h_pivot.nunique(dropna=True)
cols_24h = nunique_24h[nunique_24h > 1].index
df_24h_filtered = df_24h_pivot[cols_24h]

# Rellenar valores faltantes
df_24h_filtered = df_24h_filtered.fillna(df_24h_filtered.mean())

# Fusionar con LOS
df_modelo_24h = df_24h_filtered.merge(
    icustays_cleaned[['ICUSTAY_ID', 'LOS']], 
    on='ICUSTAY_ID', 
    how='inner'
)

In [ ]:
# Preparar datos para modelado
X = df_modelo_24h.drop(columns=['LOS', 'ICUSTAY_ID'])
y = df_modelo_24h['LOS']

# Escalar características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Datos listos para modelado:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")